In [ ]:
import pandas as pd
df=pd.read_csv('dataforbotescalationsystem.csv')
print(df.head())

In [ ]:
import nltk
nltk.download('vader_lexicon')

In [ ]:
df.dropna(inplace=True)
df['messages'] = df['messages'].str.strip()
print(df.info())

In [ ]:
def parse_conversation(text):
    turns = text.split("||")
    parsed = []
    
    for t in turns:
        if ":" in t:
            role, msg = t.split(":", 1)
            parsed.append((role.strip(), msg.strip()))
    
    return parsed

df['parsed'] = df['messages'].apply(parse_conversation)
print(df['parsed'].iloc[0])

In [44]:
def split_roles(parsed):
    user_msgs = [m for r, m in parsed if r == "User"]
    bot_msgs = [m for r, m in parsed if r == "Bot"]
    return user_msgs, bot_msgs

df['user_msgs'], df['bot_msgs'] = zip(*df['parsed'].apply(split_roles))


In [ ]:
df['num_turns'] = df['parsed'].apply(len)
df['num_user_msgs'] = df['user_msgs'].apply(len)
df['num_bot_msgs'] = df['bot_msgs'].apply(len)
df['user_bot_ratio'] = df['num_user_msgs'] / (df['num_bot_msgs'] + 1e-5)
print(df[['num_turns', 'num_user_msgs', 'num_bot_msgs', 'user_bot_ratio']].head())

In [ ]:
print(df.groupby('label')['num_turns'].mean())

In [ ]:
import nltk
nltk.download('vader_lexicon')

In [48]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def sentiment_scores(messages):
    return [sia.polarity_scores(m)['compound'] for m in messages]

df['user_sentiments'] = df['user_msgs'].apply(sentiment_scores)

In [ ]:
print(df['user_msgs'].iloc[0])
print(df['user_sentiments'].iloc[8])

In [50]:
df['avg_sentiment'] = df['user_sentiments'].apply(
    lambda x: sum(x)/len(x) if len(x) > 0 else 0
)
df['min_sentiment'] = df['user_sentiments'].apply(
    lambda x: min(x) if len(x) > 0 else 0
)
def sentiment_trend(scores):
    if len(scores) < 2:
        return 0
    return scores[-1] - scores[0]

df['sentiment_trend'] = df['user_sentiments'].apply(sentiment_trend)

In [ ]:
print(df.groupby('label')['avg_sentiment'].mean())
print(df.groupby('label')['min_sentiment'].mean())

In [ ]:
import numpy as np
import sklearn
print("Installed successfully")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def repetition_score(user_msgs):
    if len(user_msgs) < 2:
        return 0
    
    vectorizer = TfidfVectorizer().fit(user_msgs)
    vectors = vectorizer.transform(user_msgs)
    
    sim_scores = []
    
    for i in range(1, len(user_msgs)):
        sim = cosine_similarity(vectors[i], vectors[i-1])[0][0]
        sim_scores.append(sim)
    
    return max(sim_scores)  # strongest repetition

df['repetition_score'] = df['user_msgs'].apply(repetition_score)

In [53]:
fallback_phrases = [
    "i don't understand",
    "sorry",
    "can you rephrase",
    "i didn't get that",
    "unable to help",
    "not sure",
    "cannot help",
    "please clarify"
]
def fallback_count(bot_msgs):
    count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(p in msg_lower for p in fallback_phrases):
            count += 1
    
    return count

df['fallback_count'] = df['bot_msgs'].apply(fallback_count)

In [ ]:
print(df.groupby('label')['fallback_count'].mean())

In [55]:
frustration_keywords = [
    "frustrated", "angry", "not working", "still",
    "again", "why", "worst", "hate", "useless",
    "issue", "problem", "error"
]

def frustration_score(user_msgs):
    score = 0
    
    for msg in user_msgs:
        msg_lower = msg.lower()
        if any(k in msg_lower for k in frustration_keywords):
            score += 1
    
    return score

df['frustration_score'] = df['user_msgs'].apply(frustration_score)

In [56]:
generic_responses = [
    "i am here to help",
    "please clarify",
    "can you provide more details",
    "let me check",
    "i will assist you",
    "thanks for reaching out"
]

def generic_ratio(bot_msgs):
    if len(bot_msgs) == 0:
        return 0
    
    generic_count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(g in msg_lower for g in generic_responses):
            generic_count += 1
    
    return generic_count / len(bot_msgs)

df['generic_response_ratio'] = df['bot_msgs'].apply(generic_ratio)


In [ ]:
features = [
    'num_turns',
    'num_user_msgs',
    'num_bot_msgs',
    'user_bot_ratio',
    'avg_sentiment',
    'min_sentiment',
    'sentiment_trend',
    'fallback_count',
    'frustration_score',
    'generic_response_ratio'
]

X = df[features]
y = df['label']

print (X.head())
print(y.head())


In [ ]:
# Boost signals directly in existing features
X['fallback_count'] = X['fallback_count'] * 1.5  # amplify bot failures
X['frustration_score'] = X['frustration_score'] * 1.5  # amplify frustration
X['generic_response_ratio'] = X['generic_response_ratio'] * 1.2  # slightly amplify generic responses
X['sentiment_trend'] = X['sentiment_trend'].apply(lambda x: -x if x < 0 else 0)  # treat drops as positive signal

In [ ]:
# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
print(hasattr(model, "coef_"))

In [ ]:
print(df.groupby('label').mean(numeric_only=True))

In [63]:
def predict_escalation(conversation_text, model, scaler, threshold=0.5):
    
    # --- Parse ---
    parsed = parse_conversation(conversation_text)
    user_msgs, bot_msgs = split_roles(parsed)
    
    # --- Feature computation ---
    num_turns = len(parsed)
    num_user_msgs = len(user_msgs)
    num_bot_msgs = len(bot_msgs)
    user_bot_ratio = num_user_msgs / (num_bot_msgs + 1e-5)
    
    # Sentiment
    sentiments = sentiment_scores(user_msgs)
    avg_sentiment = sum(sentiments)/len(sentiments) if sentiments else 0
    min_sentiment = min(sentiments) if sentiments else 0
    sentiment_trend = sentiments[-1] - sentiments[0] if len(sentiments) > 1 else 0
    
    # Repetition
    rep_score = repetition_score(user_msgs)
    
    # Fallback
    fb_count = fallback_count(bot_msgs)
    
    # Frustration
    fr_score = frustration_score(user_msgs)
    
    # Generic responses
    gen_ratio = generic_ratio(bot_msgs)
    
    # --- Feature vector ---
    features = [[
        num_turns,
        num_user_msgs,
        num_bot_msgs,
        user_bot_ratio,
        avg_sentiment,
        min_sentiment,
        sentiment_trend,
        fb_count,
        fr_score,
        gen_ratio
    ]]
    
    # --- Scale ---
    features_scaled = scaler.transform(features)
    
    # --- Predict probability ---
    prob = model.predict_proba(features_scaled)[0][1]
    
    # --- Decision ---
    decision = 1 if prob >= threshold else 0
    
    return prob, decision

In [ ]:
print(hasattr(model, "coef_"))

In [ ]:
conv1 = """User: Hi || Bot: Hello || User: I need help with login || Bot: Sure, reset your password || User: Thanks || Bot: You're welcome"""

print(predict_escalation(conv1, model, scaler))

In [66]:
conv=""" "User: Hey there! How’s it going? || "
    "Bot: Hello! I’m good, how about you? || "
    "User: I’m fine, thanks for asking || "
    "Bot: Glad to hear that!"""

In [ ]:
print(predict_escalation(conv, model, scaler))

In [68]:
import pickle

pickle.dump(model, open("escalation_model.pkl","wb"))
pickle.dump(scaler, open("scaler.pkl","wb"))

